In [ ]:
# Install required libraries and localtunnel
!pip install -q streamlit scikit-learn pandas numpy matplotlib seaborn joblib
!npm install -g localtunnel

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

RANDOM_STATE = 42
OUTPUT_DIR = "saved_models"
MODELS_USING_SCALED_INPUT = ["Logistic Regression", "kNN"]

# 1. Prepare output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

# 3. Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# 4. Scale inputs (Fit ONLY on training data to prevent leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.pkl"))

# Save feature schema
with open(os.path.join(OUTPUT_DIR, "feature_schema.json"), "w") as f:
    json.dump({"feature_names": list(X.columns)}, f, indent=2)

# 5. Tune Decision Tree and Random Forest
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

print("--- Tuning Models via GridSearchCV ---")
grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    {"max_depth": [3, 4, 5, 6, 8, None], "min_samples_leaf": [1, 2, 3, 5]},
    cv=cv, scoring="accuracy", n_jobs=-1
).fit(X_train, y_train)

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    {"n_estimators": [100, 200], "max_depth": [5, 8, None]},
    cv=cv, scoring="accuracy", n_jobs=-1
).fit(X_train, y_train)

models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=10000),
    "Decision Tree": grid_dt.best_estimator_,
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": grid_rf.best_estimator_,
}

# 6. Train, Evaluate & Save Artifacts
def evaluate(model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    return {
        "Accuracy": round(accuracy_score(y_te, y_pred), 4),
        "AUC": round(roc_auc_score(y_te, y_proba), 4),
        "Precision": round(precision_score(y_te, y_pred), 4),
        "Recall": round(recall_score(y_te, y_pred), 4),
        "F1": round(f1_score(y_te, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_te, y_pred), 4),
    }

all_metrics = {}
for name, model in models.items():
    use_scaled = name in MODELS_USING_SCALED_INPUT
    Xtr = X_train_scaled if use_scaled else X_train
    Xte = X_test_scaled if use_scaled else X_test

    model.fit(Xtr, y_train)
    cv_scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring="accuracy")
    test_metrics = evaluate(model, Xte, y_test)

    all_metrics[name] = {
        "test_set_metrics": test_metrics,
        "cv_accuracy_mean": round(cv_scores.mean(), 4),
        "cv_accuracy_std": round(cv_scores.std(), 4),
        "scaled_input": use_scaled,
    }

    filename = f"{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(model, os.path.join(OUTPUT_DIR, filename))
    print(f"Saved {name:20s} | Test Acc: {test_metrics['Accuracy']:.4f} | CV Acc: {cv_scores.mean():.4f}")

# Save metrics JSON
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

# Save test dataset CSV for downloading/uploading
test_data = X_test.copy()
test_data["target"] = y_test.values
test_data.to_csv("test_data.csv", index=False)
print("\n[SUCCESS] All artifacts and 'test_data.csv' created successfully!")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

st.set_page_config(page_title="Classification Dashboard", layout="wide")
st.title("Machine Learning Classification Evaluation Platform")

MODEL_DIR = "saved_models"
MODELS_USING_SCALED_INPUT = ["Logistic Regression", "kNN"]

MODEL_FILES = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "kNN": "knn.pkl",
    "Naive Bayes": "naive_bayes.pkl",
    "Random Forest": "random_forest.pkl",
}

@st.cache_resource
def load_artifacts():
    scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))

    with open(os.path.join(MODEL_DIR, "feature_schema.json")) as f:
        schema = json.load(f)

    models = {
        name: joblib.load(os.path.join(MODEL_DIR, fname))
        for name, fname in MODEL_FILES.items()
    }

    metrics_path = os.path.join(MODEL_DIR, "metrics.json")
    train_metrics = None
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            train_metrics = json.load(f)

    return scaler, schema["feature_names"], models, train_metrics

try:
    scaler, feature_names, models, train_metrics = load_artifacts()
except FileNotFoundError as e:
    st.error(f"Missing artifacts in '{MODEL_DIR}/'. Run training cell first.\nMissing: {e}")
    st.stop()

st.sidebar.header("Upload Data")
uploaded_file = st.sidebar.file_uploader("Upload test_data.csv", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    st.write("### Test Dataset Preview")
    st.dataframe(df.head())

    if "target" not in df.columns:
        st.error("Dataset missing target column!")
    else:
        X = df.drop(columns=["target"])
        y = df["target"]

        missing_cols = set(feature_names) - set(X.columns)
        extra_cols = set(X.columns) - set(feature_names)
        if missing_cols:
            st.error(f"Missing expected columns: {sorted(missing_cols)}")
            st.stop()
        if extra_cols:
            st.warning(f"Ignoring extra columns: {sorted(extra_cols)}")
        X = X[feature_names]

        st.sidebar.header("Model Selection")
        selected_model = st.sidebar.selectbox("Choose Estimator", list(models.keys()))

        clf = models[selected_model]
        use_scaled = selected_model in MODELS_USING_SCALED_INPUT

        X_input = scaler.transform(X) if use_scaled else X
        preds = clf.predict(X_input)
        probs = clf.predict_proba(X_input)[:, 1]

        st.subheader(f"Evaluation Metrics: {selected_model}")
        c1, c2, c3, c4, c5, c6 = st.columns(6)
        c1.metric("Accuracy", f"{accuracy_score(y, preds):.4f}")
        c2.metric("AUC", f"{roc_auc_score(y, probs):.4f}")
        c3.metric("Precision", f"{precision_score(y, preds):.4f}")
        c4.metric("Recall", f"{recall_score(y, preds):.4f}")
        c5.metric("F1 Score", f"{f1_score(y, preds):.4f}")
        c6.metric("MCC", f"{matthews_corrcoef(y, preds):.4f}")

        if train_metrics and selected_model in train_metrics:
            tm = train_metrics[selected_model]
            st.caption(
                f"Training-time 10-fold CV accuracy: {tm['cv_accuracy_mean']:.4f} ± {tm['cv_accuracy_std']:.4f}"
            )

        col_left, col_right = st.columns(2)
        with col_left:
            st.write("### Confusion Matrix")
            fig, ax = plt.subplots(figsize=(4, 3))
            sns.heatmap(confusion_matrix(y, preds), annot=True, fmt="d", cmap="Blues", ax=ax)
            st.pyplot(fig)

        with col_right:
            st.write("### Detailed Report")
            st.dataframe(pd.DataFrame(classification_report(y, preds, output_dict=True)).T)
else:
    st.info("Upload `test_data.csv` via the sidebar to execute evaluations.")

In [ ]:
!curl https://loca.lt/mytunnelpassword

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501